## Import

In [ ]:
!pip install rasterio numpy matplotlib fiona geopandas

In [10]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

## A bit more about multiband images

In [24]:
dataset = rasterio.open('data/23-05-2020.tiff')

In [ ]:
dict(dataset.profile), dataset.bounds

You should always close your `DatasetReader object` to avoid memory leaks, “too many open files” errors, prevent file locking issues, release resources and etc.

In [31]:
dataset.close()

For this reason it is better to use `with` context manager that creates `DatasetReader object` using `read` and automatically closes the file when the block ends.

In [ ]:
# Read multiband image
with rasterio.open('data/23-05-2020.tiff') as src:
    rgb_bands = src.read((4, 3, 2)) # u define the order your read bands
    all_bands = src.read() # read all bands

    new_meta = src.meta.copy() 

    print("RGB array structure")
    print(f"Number of bands: {rgb_bands.shape[0]}")
    print(f"Image height (rows): {rgb_bands.shape[1]}")
    print(f"Image width (cols): {rgb_bands.shape[2]}")


### How to save multiband image?

In [166]:
new_meta.update({
        'count': 3
    })

with rasterio.open(
        'data/tmp_rgb.tif',
        'w',
        **new_meta
    ) as file:
    file.write(rgb_bands)


# or (more academic way)
# useful if you want to change the specific bands entity, for example.  
# or if you have bands with different resolution.

# with rasterio.open(
#         'data/tmp_rgb_academic.tif',
#         'w',
#         **new_meta
#     ) as file:
#     for count, band in enumerate(range(new_meta['count']), 1):
#         file.write(rgb_bands[band], count)

## How to clip raster data?

In [ ]:
with rasterio.open('data/17-09-2025.tiff') as src:
# with rasterio.open('data/tolyatti_17-09-2025.tif') as src:
    data = src.read()
    meta = src.meta.copy()
    print('Data shape:', data.shape)
    # print(src.meta)

    # Plot the stacked array
    plt.figure(figsize=(8, 8))
    plt.imshow(data[3, ...] * 20 / 20000, cmap="Greys_r")
    plt.title("Tolyatti, Band 4")
    plt.show()

In [ ]:
subset = data[:, 900:1400, 700:1200]
rgb_subset = np.dstack((subset[3], subset[2], subset[1]))
rgb_subset.shape

# Plot the stacked array
plt.figure(figsize=(8, 8))
plt.imshow(rgb_subset * 10 / 20000) # for image visualisation
plt.title("Tolyatti, RGB")
plt.show()

Now we can calculate the updated transormation to save out image

In [119]:
from rasterio.windows import Window
from rasterio.transform import from_bounds

# Define the window of the subset
window = Window(col_off=700, row_off=900, width=500, height=500) # From left bottom corner

# Calculate the bounds of the window
window_bounds = rasterio.windows.bounds(window, meta['transform'])

# Calculate the new transform based on the window bounds
new_transform = from_bounds(*window_bounds, window.width, window.height)

with rasterio.open(
    "data/tolyatti_clipped.tif",
    "w",
    driver="GTiff",
    height=subset.shape[1],
    width=subset.shape[2],
    count=subset.shape[0],
    dtype=subset.dtype,
    crs=meta['crs'],
    transform=new_transform,
    compress="lzw",
) as dst:
    dst.write(subset)

Another way to use `window` during data reading

In [ ]:
with rasterio.open('data/17-09-2025.tiff') as src:
    w = src.read(1, window=Window(0, 0, 512, 256)) #(col_off, row_off, width, height)

print(w.shape)


### Clipping with Vector Data

Also check how to do this using [shapefile](https://rasterio.readthedocs.io/en/latest/topics/masking-by-shapefile.html). 

In [137]:
import fiona
import rasterio.mask
import geopandas as gpd

In [ ]:
geojson_path = "/home/glados/Documents/PhD/Course FRS 2026/FRS-course/seminars/seminar_4/data/clipped_area.geojson"
bounds = gpd.read_file(geojson_path) # Geojson CRS and raster CRS SHOULD BE SAME!
# if it differs, use
# bounds = bounds.to_crs('epsg:...')

bounds['geometry'][0]

Next, apply the mask to extract only the area within the vector bounds:

In [153]:
with rasterio.open('data/17-09-2025.tiff') as src:
    out_meta = src.meta
    
    with fiona.open(geojson_path, "r") as f:
        shapes = [feature["geometry"] for feature in f]

    out_image, out_transform = rasterio.mask.mask(src, shapes, crop=True)

In [154]:
# Finally, write the clipped raster to a new file:

out_meta.update(
    {
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform,
    }
)

with rasterio.open("data/tolyatti_clipped_vector.tif", "w", **out_meta) as dst:
    dst.write(out_image)

### Mosaicing (gluing several images together)

In [164]:
import rasterio
import numpy as np
from rasterio.merge import merge

In [165]:
with rasterio.open('data/mosaic1.tiff') as src1, rasterio.open('data/mosaic2.tiff') as src2:
    meta1 = src1.meta.copy()
    mosaic, output = merge([src1, src2])


with rasterio.open(
    "data/mosaic_united.tif",
    "w",
    driver="GTiff",
    height=mosaic.shape[1],
    width=mosaic.shape[2],
    count=mosaic.shape[0],
    dtype=mosaic.dtype,
    crs=meta1['crs'],
    transform=output,
    compress="lzw",
) as dst:
    dst.write(mosaic)

### Tasks

- Prepare 3 images:
    - 1 for clipping operation
    - 2 intersecting images
- Perform 2 clipping approaches
    - Using `rasterio.windows.Window`
    - Using `rasterio.mask` (create some .geoson polygon in QGIS)
- Merge 2 intersacting images together

In [20]:
# TODO